# PCA Per Probe and ROI from `merged_dic` + `stim_df`

This notebook runs PCA using the structure you built in `PCA_dynamic_open_ephys_nwb_v2.ipynb`:
- `merged_dic`: dict of DataFrames keyed by probe (`A..F`)
- `stim_df`: event table with aligned event times

It supports:
- per-probe PCA
- ROI filtering via `in_brainRegion` (`IN_ROI` / `OUT_ROI`)
- event selection from `stim_df`


### Cell 1: Imports


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import gaussian_filter1d

sns.set_context("talk")
sns.set_style("white")


### Cell 2: Validate Required Inputs
Run this after `merged_dic` and `stim_df` exist in memory.


In [ ]:
required_vars = ["merged_dic", "stim_df"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(
        f"Missing required variables: {missing}. Run the setup cells in PCA_dynamic_open_ephys_nwb_v2.ipynb first."
    )

if not isinstance(merged_dic, dict) or len(merged_dic) == 0:
    raise ValueError("merged_dic must be a non-empty dict keyed by probe letters.")
if not isinstance(stim_df, pd.DataFrame) or stim_df.empty:
    raise ValueError("stim_df must be a non-empty DataFrame.")

print("merged_dic probes:", sorted(list(merged_dic.keys())))
print("stim_df columns:", list(stim_df.columns))


### Cell 3: User Config
- `SELECTED_PROBES`: choose probes (or `None` for all)
- `ROI_FILTER`: `IN_ROI`, `OUT_ROI`, or `None`
- event selection and labels from `stim_df`


In [ ]:
# Probe + ROI selection
SELECTED_PROBES = None      # example: ["A", "C"] ; None => all probes in merged_dic
ROI_FILTER = "IN_ROI"       # "IN_ROI", "OUT_ROI", or None

# Event selection from stim_df
EVENT_TIME_COL = "event_time_s"
EVENT_LABEL_COL = "label" if "label" in stim_df.columns else ("stimulus" if "stimulus" in stim_df.columns else None)
EVENT_FILTER_COL = EVENT_LABEL_COL   # column used to filter event types
EVENT_FILTER_VALUES = None           # example: ["baseline", "opto_epoch_1"] or ["reach"]

# Optional per-event downsample for memory/speed
MAX_EVENTS = 5000
EVENT_SUBSAMPLE_MODE = "uniform"    # "uniform" or "first"

# Spike binning/PCA parameters
WINDOW_START_S = -0.5
WINDOW_END_S = 3.5
BIN_SIZE_S = 0.05
N_COMPONENTS = 12
SMOOTH_SIGMA = 2
MAX_TENSOR_GB = 8.0


### Cell 4: Helper Functions


In [ ]:
def zscore_rows(x: np.ndarray) -> np.ndarray:
    ss = StandardScaler(with_mean=True, with_std=True)
    return ss.fit_transform(x.T).T


def normalize_probe_value(x):
    return str(x).strip().upper()[0] if pd.notna(x) and len(str(x).strip()) > 0 else np.nan


def select_events(stim_df: pd.DataFrame,
                  event_time_col: str,
                  event_label_col: str = None,
                  event_filter_col: str = None,
                  event_filter_values=None,
                  max_events: int = None,
                  subsample_mode: str = "uniform"):
    if event_time_col not in stim_df.columns:
        raise ValueError(f"{event_time_col} not found in stim_df columns: {list(stim_df.columns)}")

    out = stim_df.copy()
    out[event_time_col] = pd.to_numeric(out[event_time_col], errors="coerce")
    out = out.dropna(subset=[event_time_col]).sort_values(event_time_col).reset_index(drop=True)

    if event_filter_values is not None and event_filter_col is not None and event_filter_col in out.columns:
        keep = out[event_filter_col].astype(str).isin([str(v) for v in event_filter_values])
        out = out[keep].reset_index(drop=True)

    n_before = len(out)
    if max_events is not None and n_before > max_events:
        if subsample_mode == "first":
            idx = np.arange(max_events)
        else:
            idx = np.linspace(0, n_before - 1, max_events, dtype=int)
        out = out.iloc[idx].reset_index(drop=True)
        print(f"Downsampled events: {n_before} -> {len(out)} (mode={subsample_mode})")

    if event_label_col is not None and event_label_col in out.columns:
        labels = out[event_label_col].astype(str).to_numpy()
    elif "stimulus" in out.columns:
        labels = out["stimulus"].astype(str).to_numpy()
    else:
        labels = np.array(["all_events"] * len(out), dtype=object)

    return out, out[event_time_col].to_numpy(dtype=float), labels


def get_probe_units_df(merged_dic: dict, probe: str, roi_filter: str = None):
    if probe not in merged_dic:
        raise ValueError(f"Probe {probe} not found in merged_dic keys: {list(merged_dic.keys())}")
    df = merged_dic[probe].copy().reset_index(drop=True)

    if "spike_times" not in df.columns:
        raise ValueError(f"Probe {probe} dataframe is missing 'spike_times' column.")

    # enforce probe value (robust even if mixed)
    if "probe" in df.columns:
        df = df[df["probe"].map(normalize_probe_value) == normalize_probe_value(probe)].reset_index(drop=True)

    if roi_filter is not None:
        if "in_brainRegion" not in df.columns:
            raise ValueError("ROI filter requested but 'in_brainRegion' column is missing.")
        df = df[df["in_brainRegion"].astype(str) == str(roi_filter)].reset_index(drop=True)

    # keep units with valid spike arrays
    valid = df["spike_times"].apply(lambda x: isinstance(x, (list, np.ndarray))).to_numpy()
    df = df[valid].reset_index(drop=True)
    return df


def bin_spikes_around_events(spike_times_list, event_times_s, win_start_s, win_end_s, bin_size_s, max_tensor_gb=8.0):
    edges = np.arange(win_start_s, win_end_s + bin_size_s, bin_size_s)
    n_bins = len(edges) - 1
    n_trials = len(event_times_s)
    n_units = len(spike_times_list)

    est_gb = (n_trials * n_units * n_bins * np.dtype(np.float32).itemsize) / (1024**3)
    print(f"Requested tensor shape=({n_trials}, {n_units}, {n_bins}) est_mem={est_gb:.2f} GB")
    if est_gb > max_tensor_gb:
        raise MemoryError(
            f"Estimated tensor memory {est_gb:.2f} GB exceeds MAX_TENSOR_GB={max_tensor_gb}. "
            "Reduce events/units or increase BIN_SIZE_S."
        )

    X = np.zeros((n_trials, n_units, n_bins), dtype=np.float32)

    for u, st in enumerate(spike_times_list):
        st = np.asarray(st, dtype=float)
        if st.size == 0:
            continue
        for t, t0 in enumerate(event_times_s):
            i0 = np.searchsorted(st, t0 + win_start_s, side="left")
            i1 = np.searchsorted(st, t0 + win_end_s, side="right")
            rel = st[i0:i1] - t0
            if rel.size:
                counts, _ = np.histogram(rel, bins=edges)
                X[t, u, :] = counts / bin_size_s

    return X, edges[:-1]


def trial_level_pca(trials, labels, n_components=12):
    # trials: (n_trials, n_units, n_bins)
    X_trial = trials.mean(axis=2).T  # (n_units, n_trials)
    Xz = zscore_rows(X_trial)

    pca = PCA(n_components=min(n_components, Xz.shape[0], Xz.shape[1]))
    Xp = pca.fit_transform(Xz.T).T
    trial_types = pd.unique(labels)
    t_type_ind = [np.where(labels == t)[0] for t in trial_types]
    return Xp, pca.explained_variance_ratio_, trial_types, t_type_ind


def trajectory_pca(trials, labels, n_components=12):
    trial_types = pd.unique(labels)
    t_type_ind = [np.where(labels == t)[0] for t in trial_types]

    trial_averages = []
    kept_labels = []
    for t, idx in zip(trial_types, t_type_ind):
        if len(idx) > 0:
            trial_averages.append(trials[idx].mean(axis=0))  # (units, bins)
            kept_labels.append(t)

    if len(trial_averages) < 2:
        raise ValueError("Need at least 2 non-empty event labels for trajectory PCA.")

    Xa = np.hstack(trial_averages)
    Xaz = zscore_rows(Xa)

    pca = PCA(n_components=min(n_components, Xaz.shape[0], Xaz.shape[1]))
    Xa_p = pca.fit_transform(Xaz.T).T
    return Xa_p, pca.explained_variance_ratio_, kept_labels


def plot_trial_level_pca(Xp, trial_types, t_type_ind, title_prefix=""):
    projections = [(0, 1), (1, 2), (0, 2)]
    pal = sns.color_palette("colorblind", len(trial_types))

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (i, j) in zip(axes, projections):
        for k, t in enumerate(trial_types):
            idx = t_type_ind[k]
            ax.scatter(Xp[i, idx], Xp[j, idx], s=28, alpha=0.8, color=pal[k], label=str(t))
        ax.set_xlabel(f"PC {i+1}")
        ax.set_ylabel(f"PC {j+1}")
    axes[0].set_title(f"{title_prefix} Trial PCA")
    axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    plt.show()


def plot_trajectory_pca(Xa_p, kept_labels, n_bins, time, smooth_sigma=2, title_prefix=""):
    pal = sns.color_palette("colorblind", len(kept_labels))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
    for comp in range(min(3, Xa_p.shape[0])):
        ax = axes[comp]
        for k, lbl in enumerate(kept_labels):
            s = k * n_bins
            e = (k + 1) * n_bins
            x = Xa_p[comp, s:e]
            if smooth_sigma and smooth_sigma > 0:
                x = gaussian_filter1d(x, sigma=smooth_sigma)
            ax.plot(time, x, lw=2, color=pal[k], label=str(lbl))
        ax.axvline(0, color="gray", ls="--", lw=1)
        ax.set_ylabel(f"PC {comp+1}")

    axes[1].set_xlabel("Time from event (s)")
    axes[0].set_title(f"{title_prefix} Trajectory PCA")
    axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    plt.show()


### Cell 5: Select Events from `stim_df`
Set event filters in config, then run.


In [ ]:
events_sel_df, event_times, event_labels = select_events(
    stim_df=stim_df,
    event_time_col=EVENT_TIME_COL,
    event_label_col=EVENT_LABEL_COL,
    event_filter_col=EVENT_FILTER_COL,
    event_filter_values=EVENT_FILTER_VALUES,
    max_events=MAX_EVENTS,
    subsample_mode=EVENT_SUBSAMPLE_MODE,
)

print("Selected events:", len(events_sel_df))
print("Event labels:", pd.Series(event_labels).value_counts().head(20))


### Cell 6: Run PCA for One Probe + ROI
Set `TARGET_PROBE` and run this cell.


In [ ]:
TARGET_PROBE = "A"

probe_units_df = get_probe_units_df(
    merged_dic=merged_dic,
    probe=TARGET_PROBE,
    roi_filter=ROI_FILTER,
)

spike_times_list = [np.asarray(x, dtype=float) for x in probe_units_df["spike_times"].values]

trials_probe, time = bin_spikes_around_events(
    spike_times_list=spike_times_list,
    event_times_s=event_times,
    win_start_s=WINDOW_START_S,
    win_end_s=WINDOW_END_S,
    bin_size_s=BIN_SIZE_S,
    max_tensor_gb=MAX_TENSOR_GB,
)

print(f"Probe {TARGET_PROBE} | ROI={ROI_FILTER} | units={len(spike_times_list)} | trials={trials_probe.shape[0]}")

Xp, evr_trial, trial_types, t_type_ind = trial_level_pca(trials_probe, event_labels, n_components=N_COMPONENTS)
print("Trial PCA EVR first 5:", np.round(evr_trial[:5], 4))
plot_trial_level_pca(Xp, trial_types, t_type_ind, title_prefix=f"Probe {TARGET_PROBE} ({ROI_FILTER})")

Xa_p, evr_traj, kept_labels = trajectory_pca(trials_probe, event_labels, n_components=N_COMPONENTS)
print("Trajectory PCA EVR first 5:", np.round(evr_traj[:5], 4))
plot_trajectory_pca(Xa_p, kept_labels, n_bins=trials_probe.shape[2], time=time, smooth_sigma=SMOOTH_SIGMA,
                    title_prefix=f"Probe {TARGET_PROBE} ({ROI_FILTER})")


### Cell 7: Run PCA for Multiple Probes
Runs the same analysis probe-by-probe and stores results.


In [ ]:
if SELECTED_PROBES is None:
    probes_to_run = sorted(list(merged_dic.keys()))
else:
    probes_to_run = [p for p in SELECTED_PROBES if p in merged_dic]

results_by_probe = {}

for probe in probes_to_run:
    print("\n============================")
    print(f"Running probe {probe} | ROI={ROI_FILTER}")

    probe_units_df = get_probe_units_df(merged_dic, probe=probe, roi_filter=ROI_FILTER)
    if probe_units_df.empty:
        print(f"Skipping probe {probe}: no units after ROI/probe filtering.")
        continue

    spike_times_list = [np.asarray(x, dtype=float) for x in probe_units_df["spike_times"].values]

    try:
        trials_probe, time_probe = bin_spikes_around_events(
            spike_times_list=spike_times_list,
            event_times_s=event_times,
            win_start_s=WINDOW_START_S,
            win_end_s=WINDOW_END_S,
            bin_size_s=BIN_SIZE_S,
            max_tensor_gb=MAX_TENSOR_GB,
        )
    except MemoryError as e:
        print(f"Skipping probe {probe} due to memory guard: {e}")
        continue

    Xp, evr_trial, trial_types, t_type_ind = trial_level_pca(trials_probe, event_labels, n_components=N_COMPONENTS)
    Xa_p, evr_traj, kept_labels = trajectory_pca(trials_probe, event_labels, n_components=N_COMPONENTS)

    results_by_probe[probe] = {
        "units_df": probe_units_df,
        "trials": trials_probe,
        "time": time_probe,
        "trial_pca_scores": Xp,
        "trial_evr": evr_trial,
        "traj_pca_scores": Xa_p,
        "traj_evr": evr_traj,
        "trial_types": trial_types,
        "trial_type_indices": t_type_ind,
        "traj_labels": kept_labels,
    }

    print(f"Probe {probe}: units={len(spike_times_list)}, trials={trials_probe.shape[0]}")
    print("Trial EVR first 3:", np.round(evr_trial[:3], 4), "| Traj EVR first 3:", np.round(evr_traj[:3], 4))

print("\nCompleted probes:", list(results_by_probe.keys()))


### Cell 8: Plot Any Probe from `results_by_probe`
Use this after running multi-probe cell.


In [ ]:
PLOT_PROBE = "A"

if PLOT_PROBE not in results_by_probe:
    raise ValueError(f"{PLOT_PROBE} not in results_by_probe. Available: {list(results_by_probe.keys())}")

res = results_by_probe[PLOT_PROBE]
plot_trial_level_pca(
    Xp=res["trial_pca_scores"],
    trial_types=res["trial_types"],
    t_type_ind=res["trial_type_indices"],
    title_prefix=f"Probe {PLOT_PROBE} ({ROI_FILTER})",
)

plot_trajectory_pca(
    Xa_p=res["traj_pca_scores"],
    kept_labels=res["traj_labels"],
    n_bins=res["trials"].shape[2],
    time=res["time"],
    smooth_sigma=SMOOTH_SIGMA,
    title_prefix=f"Probe {PLOT_PROBE} ({ROI_FILTER})",
)


### Cell 9: Save Results


In [ ]:
save_prefix = f"probe_roi_pca_{ROI_FILTER if ROI_FILTER is not None else 'ALLROI'}"

# Save compact summary tables
summary_rows = []
for probe, res in results_by_probe.items():
    summary_rows.append({
        "probe": probe,
        "n_units": res["trials"].shape[1],
        "n_trials": res["trials"].shape[0],
        "trial_evr_pc1": float(res["trial_evr"][0]) if len(res["trial_evr"]) > 0 else np.nan,
        "traj_evr_pc1": float(res["traj_evr"][0]) if len(res["traj_evr"]) > 0 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / f"{save_prefix}_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
summary_df
